# 05 - Image model for the d1 subset: working experiment

This notebook trains an image-only model and saves predictions for patients kept out of its training. Its purpose is to establish an image baseline for the later comparison with metadata.

The question it answers is: can a model learn useful melanoma ranking from the skin images, and how believable are the scores it gives?

It follows the metadata experiments and checks in `04_metadata_tuning_d1.ipynb` and `04b_metadata_robustness_d1.ipynb`. It reuses the patient folds from `02_patient_splitting_d1.ipynb`. The metadata models use age, sex and lesion location; this model uses image pixels. Its saved models and predictions feed `05a_image_uncertainty_d1_wip.ipynb`, which checks how much the predictions change across repeated runs and image orientations.


## Saved-run status

The current recipe uses an EfficientNet-B0 model that starts with weights learned from ImageNet, then adapts them to melanoma images. The default prototype evaluates only fold 0. A fold is one held-out group of patients; all images from a patient stay together.

This copy of the training notebook has no saved cell outputs. The existing result files record a fold-0 run with 1,370 images from 84 patients, including 25 malignant images. Its average precision is 0.1415 and ROC-AUC is 0.8452. These values come from `results/metrics/image_model_oof_metrics_prototype_fold0_d1.csv`, rather than a fresh execution of this notebook.

The experiment saves trained models, training history, one ordinary score per held-out image, and metric tables. It generates no figures. The tables are explained at the steps where they are produced.


## How the comparison works

`image_name` identifies a lesion image. `patient_id` keeps that patient's images together. For each fold, the model trains on the other four folds and predicts the held-out patients. These are out-of-fold predictions, abbreviated OOF.

The held-out fold does not choose the architecture, number of training epochs, or saved model. The present three-epoch recipe is provisional. Any future choices or early stopping should use a separate patient split within the training data. The reserved test remains outside model development.

The saved columns use the word `probability` for a score between 0 and 1. The training loss gives malignant images extra weight, which can change the score scale. Until calibration is checked, a score of 0.70 should not be read as a 70% melanoma risk.


## Setup

### Step 1 - Choose the run settings

`RUN_MODE = "prototype"` runs fold 0 so the training workflow can be checked while the recipe is still changing. `"full_oof"` runs all five folds after the recipe has been fixed. Set `TRAIN_MODELS = True` only when intending to train and replace the selected fold models.

If GPU memory is insufficient, a smaller batch size of 64 or 32 may allow training to run. Record changes to the training settings and rebuild later outputs so the notebooks describe the same model.


In [ ]:
import json
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from IPython.display import display
from PIL import Image
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0
from tqdm.auto import tqdm


# Change this one path if the project is stored somewhere else.
PROJECT_ROOT = Path(
    r"C:\Users\thoma\Documents\GitHub\ADS2002-Melanoma"
).resolve()

DATA_PATH = PROJECT_ROOT / "data" / "subset" / "processed" / "train_folds.csv"
TRAIN_IMAGE_FOLDER = PROJECT_ROOT / "data" / "subset" / "train"

RESULTS_FOLDER = PROJECT_ROOT / "results"
PREDICTIONS_FOLDER = RESULTS_FOLDER / "predictions"
METRICS_FOLDER = RESULTS_FOLDER / "metrics"
MODEL_FOLDER = PROJECT_ROOT / "models" / "image_d1_wip"

RUN_MODE = "prototype"
if RUN_MODE not in {"prototype", "full_oof"}:
    raise ValueError("RUN_MODE must be either 'prototype' or 'full_oof'.")

FOLDS_TO_RUN = [0] if RUN_MODE == "prototype" else [0, 1, 2, 3, 4]
RUN_TAG = "prototype_fold0" if RUN_MODE == "prototype" else "full_oof"

MODEL_NAME = "efficientnet_b0_wip"
RANDOM_SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 128
NUMBER_OF_EPOCHS = 3
CLASSIFIER_LEARNING_RATE = 1e-3
FINE_TUNING_LEARNING_RATE = 1e-4

# True trains and overwrites the selected fold checkpoints.
# False reuses checkpoints that already match this configuration.
TRAIN_MODELS = True

BASE_OOF_PATH = (
    PREDICTIONS_FOLDER / f"image_model_oof_{RUN_TAG}_d1.csv"
)
TRAINING_HISTORY_PATH = (
    METRICS_FOLDER / f"image_model_training_history_{RUN_TAG}_d1.csv"
)
OOF_METRICS_PATH = (
    METRICS_FOLDER / f"image_model_oof_metrics_{RUN_TAG}_d1.csv"
)
RUN_CONFIG_PATH = (
    METRICS_FOLDER / f"image_model_run_config_{RUN_TAG}_d1.json"
)

for output_folder in [PREDICTIONS_FOLDER, METRICS_FOLDER, MODEL_FOLDER]:
    output_folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Run mode: {RUN_MODE}; folds: {FOLDS_TO_RUN}")
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Platform: {platform.platform()}")

### Step 2 - Set the random seed and select the device

A seed controls the random choices during training, which makes repeated runs easier to compare. It does not remove all hardware or software differences. The code uses a compatible GPU when available and otherwise uses the CPU.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed_everything(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU. Training will be considerably slower.")
    if torch.version.cuda is None:
        print(
            "This environment has a CPU-only PyTorch build. Install a "
            "CUDA-enabled PyTorch build in this kernel, then restart it."
        )
    else:
        print(
            f"PyTorch was compiled for CUDA {torch.version.cuda}, but the GPU "
            "is unavailable. Check the driver and selected Jupyter kernel."
        )

## Data

### Step 3 - Check the patient folds and image files

Before training, the code checks the required columns, image and patient identifiers, labels, fold membership, and image file paths. These checks help prevent a missing image or a patient appearing in two folds from silently changing the comparison.

Read the fold table one row at a time. `images` counts lesion images, `patients` counts people, and `malignant_images` counts images labelled melanoma. `malignancy_rate` is the fraction of images labelled melanoma in that fold. Similar rates help the folds stay comparable; the table alone cannot establish that every aspect of the patient mix is balanced.


In [ ]:
required_paths = [DATA_PATH, TRAIN_IMAGE_FOLDER]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required project inputs are missing: "
        + ", ".join(str(path) for path in missing_paths)
    )

train = pd.read_csv(DATA_PATH)

required_columns = {"image_name", "patient_id", "target", "fold"}
missing_columns = required_columns - set(train.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

if train.empty:
    raise ValueError("The training fold file is empty.")
if train["image_name"].isna().any() or train["image_name"].duplicated().any():
    raise ValueError("image_name must be complete and unique.")
if train["patient_id"].isna().any():
    raise ValueError("patient_id contains missing values.")
if set(train["target"].unique()) != {0, 1}:
    raise ValueError("target must contain both binary classes 0 and 1.")
if set(train["fold"].unique()) != {0, 1, 2, 3, 4}:
    raise ValueError("Expected five folds labelled 0 through 4.")
if train.groupby("patient_id")["fold"].nunique().gt(1).any():
    raise ValueError("At least one patient appears in multiple folds.")

train["image_path"] = train["image_name"].map(
    lambda image_name: TRAIN_IMAGE_FOLDER / f"{image_name}.jpg"
)
missing_images = train.loc[
    ~train["image_path"].map(Path.exists), "image_name"
].tolist()
if missing_images:
    raise FileNotFoundError(
        f"{len(missing_images)} training images are missing; "
        f"examples: {missing_images[:10]}"
    )

fold_summary = (
    train.groupby("fold")
    .agg(
        images=("image_name", "size"),
        patients=("patient_id", "nunique"),
        malignant_images=("target", "sum"),
        malignancy_rate=("target", "mean"),
    )
    .reset_index()
)

print(f"Images: {len(train):,}")
print(f"Patients: {train['patient_id'].nunique():,}")
print(
    f"Malignant images: {int(train['target'].sum()):,} "
    f"({train['target'].mean():.2%})"
)
display(fold_summary.style.format({"malignancy_rate": "{:.2%}"}))

The saved fold file defines which patients are held out. This stage uses the development data and does not evaluate the reserved test.


## Image preparation

### Step 4 - Prepare the training and held-out images

Training randomly flips and rotates images so the model has less reason to rely on one orientation. Every image is resized to 224 by 224 pixels and put on the numeric scale expected by the pretrained model.

The ordinary held-out prediction uses one fixed image preparation. Keeping that preparation fixed makes the score repeatable and gives the next notebook a clear starting point for its repeated-prediction checks.


In [ ]:
normalise = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    normalise,
])

validation_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalise,
])

print("Training transformation: random flips and rotation.")
print("Validation transformation: resize, tensor conversion, and normalisation.")

The random image changes are used during training. The ordinary held-out score uses the fixed version of the image.


## Model

### Step 5 - Build the image model

The dataset reads one image at a time. EfficientNet-B0 starts with image features learned from ImageNet, and the code replaces its final classifier with a single output for melanoma. A sigmoid later converts that output into a score between 0 and 1.

B0 is a manageable starting model for a laptop GPU. This experiment does not establish that B0 is the best architecture; that would need a fair comparison with alternatives using training patients.


In [ ]:
class MelanomaDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["image_path"]) as source_image:
            image = source_image.convert("RGB")
            image = self.transform(image)

        return {
            "image": image,
            "target": torch.tensor(row["target"], dtype=torch.float32),
            "image_id": row["image_name"],
            "patient_id": row["patient_id"],
            "fold": torch.tensor(row["fold"], dtype=torch.int64),
        }


def build_model(freeze_features=True, pretrained=True):
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    number_of_inputs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(number_of_inputs, 1)

    if freeze_features:
        for parameter in model.features.parameters():
            parameter.requires_grad = False

    return model.to(device)


def make_loader(dataframe, transform, shuffle, seed):
    dataset = MelanomaDataset(dataframe, transform)
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )


def checkpoint_path_for_fold(fold):
    return MODEL_FOLDER / f"efficientnet_b0_wip_fold_{fold}.pth"

### Step 6 - Define the training recipe and saved-model checks

One epoch means a pass through the training images. During epoch 1, the code updates only the new classifier's weights. From epoch 2, it updates the full network with a smaller learning rate. It uses the same three-epoch recipe for every fold.

The loss gives each malignant image more influence because there are far fewer malignant than benign images. This may help the model learn to rank melanoma cases, but it also means the score scale needs a calibration check.

The saved-model checks compare basic settings such as the fold, image size, epoch count, and seed. They do not capture the full data and training history; the limitations below explain what this leaves uncertain.


In [ ]:
def binary_probability_metrics(targets, probabilities):
    targets = np.asarray(targets, dtype=float)
    probabilities = np.asarray(probabilities, dtype=float)

    if not np.isfinite(probabilities).all():
        raise ValueError("Probabilities contain non-finite values.")
    if ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Probabilities must lie between 0 and 1.")

    return {
        "roc_auc": float(roc_auc_score(targets, probabilities)),
        "average_precision": float(
            average_precision_score(targets, probabilities)
        ),
        "log_loss": float(log_loss(targets, probabilities, labels=[0, 1])),
        "brier_score": float(brier_score_loss(targets, probabilities)),
        "mean_absolute_probability_error": float(
            np.mean(np.abs(targets - probabilities))
        ),
    }


def train_fixed_recipe(model, training_loader, positive_weight, fold):
    positive_weight_tensor = torch.tensor(
        [positive_weight], dtype=torch.float32, device=device
    )
    loss_function = nn.BCEWithLogitsLoss(pos_weight=positive_weight_tensor)
    optimizer = torch.optim.AdamW(
        model.classifier.parameters(),
        lr=CLASSIFIER_LEARNING_RATE,
    )
    fold_history = []

    for epoch_index in range(NUMBER_OF_EPOCHS):
        epoch = epoch_index + 1

        if epoch == 2:
            for parameter in model.parameters():
                parameter.requires_grad = True
            optimizer = torch.optim.AdamW(
                model.parameters(),
                lr=FINE_TUNING_LEARNING_RATE,
            )

        model.train()
        epoch_start = time.time()
        total_loss = 0.0

        for batch in tqdm(
            training_loader,
            desc=f"Fold {fold} epoch {epoch}/{NUMBER_OF_EPOCHS}",
        ):
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["target"].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images).squeeze(1)
            loss = loss_function(logits, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        fold_history.append({
            "fold": int(fold),
            "epoch": int(epoch),
            "training_loss": total_loss / len(training_loader),
            "epoch_minutes": (time.time() - epoch_start) / 60,
        })

    return pd.DataFrame(fold_history)


def save_checkpoint(model, fold, positive_weight):
    path = checkpoint_path_for_fold(fold)
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "fold": int(fold),
            "image_size": int(IMAGE_SIZE),
            "epochs": int(NUMBER_OF_EPOCHS),
            "random_seed": int(RANDOM_SEED),
            "positive_weight": float(positive_weight),
        },
        path,
    )
    return path


def load_checkpoint(fold):
    path = checkpoint_path_for_fold(fold)
    if not path.exists():
        raise FileNotFoundError(f"Missing fold checkpoint: {path}")

    checkpoint = torch.load(path, map_location=device, weights_only=True)
    expected_values = {
        "model_name": MODEL_NAME,
        "fold": int(fold),
        "image_size": int(IMAGE_SIZE),
        "epochs": int(NUMBER_OF_EPOCHS),
        "random_seed": int(RANDOM_SEED),
    }
    for field, expected_value in expected_values.items():
        if checkpoint.get(field) != expected_value:
            raise ValueError(
                f"Checkpoint {path} has {field}={checkpoint.get(field)!r}; "
                f"expected {expected_value!r}. Retrain this fold."
            )

    model = build_model(freeze_features=False, pretrained=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint

The held-out fold is evaluated after training and does not choose the saved model. If early stopping is added, choose the stopping point using separate patients inside the training folds.


### Step 7 - Predict the held-out images

Each image receives one score using the fixed preparation and the saved model for its fold. Higher scores place an image nearer the top of the model's melanoma ranking. The score scale still needs checking before it can be interpreted as risk.


In [ ]:
def predict_probabilities(model, dataframe, transform, seed):
    loader = make_loader(
        dataframe,
        transform=transform,
        shuffle=False,
        seed=seed,
    )
    image_ids = []
    probabilities = []

    model.eval()
    with torch.inference_mode():
        for batch in tqdm(loader, leave=False, desc="Validation inference"):
            images = batch["image"].to(device, non_blocking=True)
            logits = model(images).squeeze(1)
            batch_probabilities = torch.sigmoid(logits).cpu().numpy()

            image_ids.extend(batch["image_id"])
            probabilities.extend(batch_probabilities.astype(float))

    return image_ids, np.asarray(probabilities, dtype=float)

### Step 8 - Train the selected folds and collect predictions

For each selected fold, the code trains on the other four folds, checks that the patient groups do not overlap, saves or reloads the model, and predicts the held-out images. It also records the training loss and time for each epoch.

A lower training loss means the model fits its training objective better. It does not show how well the model will work on new patients; the held-out metrics answer that part.


In [ ]:
oof_parts = []
training_history_parts = []

for fold in FOLDS_TO_RUN:
    print(f"\n{'=' * 72}\nOuter fold {fold}\n{'=' * 72}")
    training_frame = train.loc[train["fold"].ne(fold)].copy()
    validation_frame = train.loc[train["fold"].eq(fold)].copy()

    patient_overlap = set(training_frame["patient_id"]).intersection(
        set(validation_frame["patient_id"])
    )
    if patient_overlap:
        raise RuntimeError(
            f"Fold {fold} has {len(patient_overlap)} shared patients."
        )

    number_positive = int(training_frame["target"].sum())
    number_negative = len(training_frame) - number_positive
    positive_weight = number_negative / number_positive

    print(
        f"Training images: {len(training_frame):,}; "
        f"held-out images: {len(validation_frame):,}; "
        f"positive weight: {positive_weight:.3f}"
    )

    if TRAIN_MODELS:
        seed_everything(RANDOM_SEED + int(fold))
        model = build_model(freeze_features=True)
        training_loader = make_loader(
            training_frame,
            transform=train_transform,
            shuffle=True,
            seed=RANDOM_SEED + int(fold),
        )
        fold_history = train_fixed_recipe(
            model,
            training_loader,
            positive_weight=positive_weight,
            fold=fold,
        )
        checkpoint_path = save_checkpoint(
            model,
            fold=fold,
            positive_weight=positive_weight,
        )
        training_history_parts.append(fold_history)
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Saved checkpoint: {checkpoint_path}")

    model, checkpoint = load_checkpoint(fold)
    validation_frame = validation_frame.reset_index(drop=True)
    image_ids, probabilities = predict_probabilities(
        model,
        validation_frame,
        transform=validation_transform,
        seed=RANDOM_SEED + 1000 * int(fold),
    )

    expected_ids = validation_frame["image_name"].tolist()
    if image_ids != expected_ids:
        raise RuntimeError(f"Image order changed in fold {fold}.")

    fold_predictions = validation_frame[
        ["image_name", "patient_id", "fold", "target"]
    ].rename(columns={"image_name": "image_id"})
    fold_predictions["model_name"] = MODEL_NAME
    fold_predictions["base_probability"] = probabilities
    oof_parts.append(fold_predictions)

    fold_metrics = binary_probability_metrics(
        fold_predictions["target"],
        fold_predictions["base_probability"],
    )
    print(
        f"Fold {fold} AP={fold_metrics['average_precision']:.4f}; "
        f"ROC-AUC={fold_metrics['roc_auc']:.4f}; "
        f"Brier={fold_metrics['brier_score']:.4f}"
    )

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

oof_predictions = pd.concat(oof_parts, ignore_index=True)

if training_history_parts:
    training_history = pd.concat(training_history_parts, ignore_index=True)
else:
    training_history = pd.DataFrame(
        columns=["fold", "epoch", "training_loss", "epoch_minutes"]
    )

## Results

### Step 9 - Check and save the handoff file

The code checks that every selected held-out image appears once and that its score is between 0 and 1. It then saves the rows used by the uncertainty notebook.

The preview table shows which image and patient each score belongs to. `target = 1` means the image is labelled malignant, and `target = 0` means benign. `base_probability` is the ordinary image-model score. A handful of preview rows cannot describe overall model performance.


In [ ]:
expected_rows = int(train["fold"].isin(FOLDS_TO_RUN).sum())
expected_ids = set(
    train.loc[train["fold"].isin(FOLDS_TO_RUN), "image_name"]
)

if len(oof_predictions) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} OOF rows; found {len(oof_predictions)}."
    )
if set(oof_predictions["image_id"]) != expected_ids:
    raise RuntimeError("OOF image identifiers do not match the selected folds.")
if oof_predictions["image_id"].duplicated().any():
    raise RuntimeError("At least one image has multiple OOF rows.")
if oof_predictions.groupby("patient_id")["fold"].nunique().gt(1).any():
    raise RuntimeError("At least one patient appears in multiple OOF folds.")
if oof_predictions["base_probability"].isna().any():
    raise RuntimeError("At least one image probability is missing.")
if not oof_predictions["base_probability"].between(0, 1).all():
    raise RuntimeError("Image probabilities must lie between 0 and 1.")

oof_predictions = oof_predictions.sort_values(
    ["fold", "image_id"]
).reset_index(drop=True)
oof_predictions.to_csv(BASE_OOF_PATH, index=False)
training_history.to_csv(TRAINING_HISTORY_PATH, index=False)

run_config = {
    "run_mode": RUN_MODE,
    "folds_run": [int(fold) for fold in FOLDS_TO_RUN],
    "model_name": MODEL_NAME,
    "random_seed": RANDOM_SEED,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": NUMBER_OF_EPOCHS,
    "classifier_learning_rate": CLASSIFIER_LEARNING_RATE,
    "fine_tuning_learning_rate": FINE_TUNING_LEARNING_RATE,
    "training_transform": "resize, random horizontal/vertical flip, random rotation",
    "validation_transform": "resize, tensor conversion, ImageNet normalisation",
    "uncertainty_calculated": False,
    "test_set_loaded": False,
}
RUN_CONFIG_PATH.write_text(
    json.dumps(run_config, indent=2), encoding="utf-8"
)

print(f"Saved image probabilities: {BASE_OOF_PATH}")
print(f"Saved training history: {TRAINING_HISTORY_PATH}")
print(f"OOF coverage: {len(oof_predictions):,}/{len(train):,} images")
display(oof_predictions.head())

Prototype mode covers fold 0 only. The coverage count should therefore be read as a workflow check, rather than a completed five-fold comparison. Complete OOF results require all five held-out folds.


### Step 10 - Read the metric table

Average precision, or AP, measures how effectively malignant images rise to the top of the ranking as the score cutoff changes. Higher is better. Because melanoma is rare, compare AP with the fraction of malignant images as a rough no-skill reference. ROC-AUC also measures ranking; 0.5 is a chance-level reference and 1 is perfect ranking. Neither metric measures the fraction of all images classified correctly.

Log loss, Brier score, and mean absolute probability error compare the score with the known label. Lower is better. They check the score scale as well as its ranking. Mean absolute error is strongly influenced by the many benign images, so it must be read alongside the other metrics.

The `partial_oof` row pools the selected held-out folds. In prototype mode, it repeats the same result as the fold-0 row. These are two views of one evaluation, rather than independent results.


In [ ]:
metric_rows = []

overall_scope = (
    "full_oof" if set(FOLDS_TO_RUN) == {0, 1, 2, 3, 4}
    else "partial_oof"
)
metric_rows.append({
    "scope": overall_scope,
    "fold": pd.NA,
    "images": len(oof_predictions),
    "patients": oof_predictions["patient_id"].nunique(),
    "malignant_images": int(oof_predictions["target"].sum()),
    **binary_probability_metrics(
        oof_predictions["target"],
        oof_predictions["base_probability"],
    ),
})

for fold in sorted(oof_predictions["fold"].unique()):
    fold_rows = oof_predictions["fold"].eq(fold)
    metric_rows.append({
        "scope": "fold",
        "fold": int(fold),
        "images": int(fold_rows.sum()),
        "patients": oof_predictions.loc[fold_rows, "patient_id"].nunique(),
        "malignant_images": int(
            oof_predictions.loc[fold_rows, "target"].sum()
        ),
        **binary_probability_metrics(
            oof_predictions.loc[fold_rows, "target"],
            oof_predictions.loc[fold_rows, "base_probability"],
        ),
    })

oof_metrics = pd.DataFrame(metric_rows)
oof_metrics.to_csv(OOF_METRICS_PATH, index=False)

print(f"Saved image-model metrics: {OOF_METRICS_PATH}")
display(oof_metrics.round(5))

## Limitations

The saved prototype covers one fold, with 25 malignant images from 84 patients. It cannot establish stable performance across the whole development dataset, select the best architecture, or prove that images improve on metadata. Comparing this one-fold result with a metadata score pooled over five folds would use different evaluation populations.

The three-epoch training recipe has not been selected through a patient-safe comparison. In the first epoch, the feature weights are held fixed, but BatchNorm's stored statistics can still update because the model remains in training mode. This is weaker than fully freezing every part of the feature extractor.

The saved prototype uses class-weighted training and has poorly calibrated scores. The mean score is 42.3%, while 1.8% of the held-out images are malignant. The next notebook's reliability figure examines this mismatch. These scores should not yet be presented as a patient's melanoma risk.

Saved models record only part of the training settings and do not identify the exact data or fold file used. Loading an old model after the data, folds, or recipe change can invalidate the held-out comparison. Retrain and regenerate the later outputs when those inputs change. In prediction-only mode, the current save step also writes an empty training-history file, so preserve the original history before using that mode.

No uncertainty interval, clinical decision rule, external dataset evaluation, or final reserved-test result is established here.


## Summary

The saved fold-0 prototype shows useful ranking: AP is 0.1415 and ROC-AUC is 0.8452 on 1,370 held-out images. Its Brier score of 0.2294 and the mismatch between average score and melanoma frequency warn that the scores need calibration work.

Keep EfficientNet-B0 as a development baseline. This evidence is too limited to select it as the final model. Next, use training patients to settle the training recipe, address the saved-model and score-scale limitations, and generate all five held-out folds. Then rerun `05a_image_uncertainty_d1_wip.ipynb` with those matching models and predictions.
